# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Connect to the warehouse

Same Hugging Face + DuckDB connection as Week 4 — reused unchanged so this notebook can run independently, without needing w04_baseline_score.ipynb open in the same session

In [4]:

%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print('connected')

connected


Week-4 rule constants (unchanged)

In [5]:
WINDOW_DAYS       = 15
MIN_PREV_IMPR     = 50
IMPR_DROP_THRESH  = 0.20
CLICK_DROP_THRESH = 0.20
POS_SLIP_THRESH   = 1.0
ACTIVITY_DROP_FRAC= 0.25

W_IMPR, W_CLICK, W_POS, W_ACTIVITY = 0.40, 0.25, 0.20, 0.15
assert abs((W_IMPR + W_CLICK + W_POS + W_ACTIVITY) - 1.0) < 1e-9

REASON_CODES = ['IMPR_DROP', 'CLICK_DROP', 'POSITION_SLIP', 'ACTIVITY_DROP']
print('rule constants set')


rule constants set


In [8]:
windowed = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS decision_day FROM {TABLES['fact_daily']}
    ),
    per_item AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_impressions ELSE 0 END)                              AS imp_last,
            SUM(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_impressions ELSE 0 END)                              AS imp_prev,
            SUM(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_clicks ELSE 0 END)                                   AS clk_last,
            SUM(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_clicks ELSE 0 END)                                   AS clk_prev,
            AVG(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_avg_position END)                                    AS pos_last,
            AVG(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_avg_position END)                                    AS pos_prev,
            COUNT(DISTINCT CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     AND f.gsc_impressions > 0 THEN f.report_date END)               AS active_days_last,
            COUNT(DISTINCT CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     AND f.gsc_impressions > 0 THEN f.report_date END)               AS active_days_prev,
            MAX(b.decision_day)                                                      AS decision_day
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.decision_day - INTERVAL ({2*WINDOW_DAYS}) DAY
          AND f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT * FROM per_item
    WHERE imp_prev >= {MIN_PREV_IMPR}
""").df()

d = windowed.copy()
d['pct_impr_change']  = (d['imp_last'] - d['imp_prev']) / d['imp_prev']
d['pct_click_change'] = np.where(d['clk_prev'] > 0, (d['clk_last'] - d['clk_prev']) / d['clk_prev'], np.nan)
d['pos_change']       = d['pos_last'] - d['pos_prev']
d['activity_change']  = (d['active_days_prev'] - d['active_days_last']) / d['active_days_prev'].clip(lower=1)

d['impr_drop_score']     = (-d['pct_impr_change']).clip(lower=0, upper=1)
d['click_drop_score']    = (-d['pct_click_change']).clip(lower=0, upper=1).fillna(0)
d['pos_slip_score']      = (d['pos_change'] / 5.0).clip(lower=0, upper=1)
d['activity_drop_score'] = d['activity_change'].clip(lower=0, upper=1)

def reasons(row):
    fired = []
    if row['pct_impr_change'] <= -IMPR_DROP_THRESH: fired.append('IMPR_DROP')
    if pd.notna(row['pct_click_change']) and row['pct_click_change'] <= -CLICK_DROP_THRESH: fired.append('CLICK_DROP')
    if row['pos_change'] >= POS_SLIP_THRESH: fired.append('POSITION_SLIP')
    if row['activity_change'] >= ACTIVITY_DROP_FRAC: fired.append('ACTIVITY_DROP')
    return fired

d['reason_codes'] = d.apply(reasons, axis=1)
d['n_reasons']     = d['reason_codes'].apply(len)

print(f'{len(d):,} total scored items | {(d["n_reasons"]>0).sum():,} flagged by the rule')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

99,893 total scored items | 78,630 flagged by the rule


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [9]:

print("Section 1 is markdown-only — see cells above/below.")

Section 1 is markdown-only — see cells above/below.


### Finding 1: The Freshness Multiplier (Finding #4 / #8)

**Claim:** "365+ day content that was refreshed within 30 days shows a 3.2x health boost
(from 10.7 to 34.5) and 57x more impressions (from 71 to 4,039)."

**Where does the label come from?** Both outcome metrics are drawn from the *local
active-content feature-vector sample*, which the paper's own Study Scope section defines
as pages with `impressions_90d > 0 AND sessions_90d > 0` — i.e. the sample only includes
pages that were still generating traffic at the time of the snapshot.

**My methodology question:** Does the "365+, unrefreshed" comparison group suffer from
survivorship bias? A 365+ day page that decayed all the way to zero impressions would be
excluded from the active-content sample entirely (it fails the `impressions_90d > 0`
filter), while a page that got refreshed and *therefore* regained impressions would
qualify. If that's the mechanism, part of the 57x gap could be a sampling artifact of
which pages are still eligible to be measured, rather than purely a refresh effect. I'd
ask: was the "no refresh" comparison group also required to be active at *both* the
start and end of the measurement window, or only at the end? The paper is admirably
careful about the small `361+` freshness bucket (flagging it as unstable at n=1), so I'd
guess this same discipline was applied here — just worth confirming, since it's the
paper's single most-repeated headline number.

---

### Finding 2: Growth Prediction (ML Appendix — Logistic Regression, 71% holdout accuracy)

**Claim:** A logistic regression trained on content age, days since update, days visible,
and other features achieves 71% holdout accuracy separating growing from declining pages,
with content age as the strongest negative coefficient.

**Where does the label come from?** The growing/declining label appears to come from the
same portfolio-level impression trend used in Finding #1 (30d-vs-prev-30d direction), and
the features (age, days since update, days visible) are computed from the same underlying
content records across 57 brands.

**My methodology question:** Was the 80/20 holdout split done at the row (page) level or
grouped by brand? The Methodology section states "Random Forest (80/20 split)... Logistic
Regression (80/20 split)" without specifying grouping. With 57 brands contributing
unevenly to 341,701 pages, a row-level split could let pages from the same brand appear
in both train and test, letting the model partly learn brand-specific baseline patterns
(a brand's typical publishing cadence, CMS behavior, or seasonal traffic) rather than a
generalizable content-lifecycle signal. This is exactly the failure mode we audited in
our own Week-5 model (Section 2 below) — I'd ask whether a brand-grouped re-run of this
71% number holds up, the same way we tested it on our own model.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


Week 5's Logistic Regression already used `GroupShuffleSplit` on `client_hash_id` — the
honest split for this data, since content items from the same client share site-wide
events (a migration, a Core Update, a CMS outage) that would leak across a naive
row-level split.

**Before/after, done properly:** rather than re-stating the Week-5 result, this section
trains and evaluates the identical model twice on the identical data — once under a
**dishonest row-level random split** (what a less careful pass might have used), and once
under the **honest grouped split** already used in Week 5 — to show concretely how much
the row-level split *overstates* performance. The gap between the two is the audit.

### Rebuild the Week-5 target and features
`d` (the Week-4 scored population) already exists from the setup cells above. This
recreates `d2` — the labeled frame used to train the Week-5 model — fresh in this
notebook, so Section 2 doesn't depend on `w05_model.ipynb` having been run first.

In [14]:
# This cell is for CODE (numbers, a query, a check).
FEATURES = ['pct_impr_change', 'pct_click_change', 'pos_change', 'activity_change', 'log_imp_prev']

d2 = d.copy()
d2['high_confidence_decline'] = (
    (d2['n_reasons'] > 0) & (d2['imp_prev'] >= MIN_PREV_IMPR * 5)
).astype(int)
d2['log_imp_prev'] = np.log1p(d2['imp_prev'])

# Drop rows with NaN in any feature or the target — pct_click_change is NaN whenever
# clk_prev == 0, and LogisticRegression cannot fit on NaN values.
before_drop = len(d2)
d2 = d2.dropna(subset=FEATURES + ['high_confidence_decline']).reset_index(drop=True)
print(f"dropped {before_drop - len(d2):,} rows with NaN features | {len(d2):,} rows remain")

print(d2['high_confidence_decline'].value_counts(normalize=True).round(4))

dropped 41,226 rows with NaN features | 58,667 rows remain
high_confidence_decline
1    0.5908
0    0.4092
Name: proportion, dtype: float64


In [15]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

SEED = 42

def precision_at_k(df, score_col, k=20):
    return df.sort_values(score_col, ascending=False).head(k)['high_confidence_decline'].mean()

def fit_eval(train_df, test_df, label):
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
    clf.fit(train_df[FEATURES], train_df['high_confidence_decline'])
    test_df = test_df.copy()
    test_df['model_score'] = clf.predict_proba(test_df[FEATURES])[:, 1]
    pr_auc = average_precision_score(test_df['high_confidence_decline'], test_df['model_score'])
    p20 = precision_at_k(test_df, 'model_score', 20)
    print(f"{label:35s}  PR-AUC={pr_auc:.3f}  precision@20={p20:.3f}")
    return pr_auc, p20

# --- BEFORE: dishonest row-level split (no grouping — a client can appear in both sides) ---
train_row, test_row = train_test_split(d2, test_size=0.2, random_state=SEED, stratify=d2['high_confidence_decline'])
row_overlap = set(train_row['client_hash_id']) & set(test_row['client_hash_id'])
print(f"Row-level split client overlap: {len(row_overlap)} clients appear on BOTH sides")
before_pr_auc, before_p20 = fit_eval(train_row, test_row, "BEFORE (row-level split)")

# --- AFTER: honest grouped split (same as Week 5) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(d2, groups=d2['client_hash_id']))
train_grp, test_grp = d2.iloc[tr_idx], d2.iloc[te_idx]
grp_overlap = set(train_grp['client_hash_id']) & set(test_grp['client_hash_id'])
print(f"Grouped split client overlap: {len(grp_overlap)} clients appear on BOTH sides")
after_pr_auc, after_p20 = fit_eval(train_grp, test_grp, "AFTER (grouped split, Week-5's choice)")

print(f"\nPR-AUC gap (before - after):        {before_pr_auc - after_pr_auc:+.3f}")
print(f"precision@20 gap (before - after):  {before_p20 - after_p20:+.3f}")

Row-level split client overlap: 42 clients appear on BOTH sides
BEFORE (row-level split)             PR-AUC=0.868  precision@20=1.000
Grouped split client overlap: 0 clients appear on BOTH sides
AFTER (grouped split, Week-5's choice)  PR-AUC=0.921  precision@20=1.000

PR-AUC gap (before - after):        -0.054
precision@20 gap (before - after):  +0.000


**Observed result:** [fill in from output — state whether the row-level split produced a
higher or lower PR-AUC / precision@20 than the grouped split, and by how much]. [If
row-level scored higher: this is the expected direction — a split that leaks client-level
information gives the model an unfair preview of test-set patterns, inflating its
apparent performance. The grouped split's lower number is the more honest estimate of how
this model would perform on a genuinely new client.] Report the actual observed numbers
here rather than assuming the direction before running.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# 1. No feature is mechanically derived from the target itself.
target_derived_terms = ['n_reasons', 'reason_codes', 'action_score', 'primary_reason', 'high_confidence_decline']
leaked_in_features = [f for f in FEATURES if f in target_derived_terms]
print('Features that ARE the target/rule output:', leaked_in_features if leaked_in_features else 'none — clean')

# 2. No feature uses data from AFTER decision_day (both windows in `d` are already
#    fully in the past relative to decision_day; confirm no forward-looking column exists).
suspect_cols = [c for c in d2.columns if 'future' in c.lower() or 'next' in c.lower() or 'outcome' in c.lower()]
print('Forward-looking column names found:', suspect_cols if suspect_cols else 'none — clean')

# 3. fact_query_90d was never joined in (same exclusion reasoning as Week 4/5 — its
#    90-day trailing window isn't confirmed to align with decision_day).
q90d_cols = ['visible_queries', 'rare_share', 'anon_share', 'top_query_share']
print('fact_query_90d columns present:', [c for c in q90d_cols if c in d2.columns] or 'none — correctly excluded')

# 4. Correlation check: does any feature correlate suspiciously close to 1.0 with the
#    target, which would suggest it's a near-duplicate encoding of the label rather than
#    an independent signal?
corr_with_target = d2[FEATURES + ['high_confidence_decline']].corr()['high_confidence_decline'].drop('high_confidence_decline')
print('\nFeature correlation with target:\n', corr_with_target.sort_values())
print('\nAny feature above 0.95 correlation (near-duplicate of label):',
      corr_with_target[corr_with_target.abs() > 0.95].index.tolist() or 'none')

# 5. Group leakage: confirm client_hash_id and content_hash_id themselves are excluded
#    from FEATURES — an ID column can act as a leakage shortcut if the model memorizes
#    per-client base rates instead of learning the general signal.
id_leak = [f for f in FEATURES if 'hash_id' in f]
print('\nID columns present in FEATURES:', id_leak if id_leak else 'none — clean')

Features that ARE the target/rule output: none — clean
Forward-looking column names found: none — clean
fact_query_90d columns present: none — correctly excluded

Feature correlation with target:
 pct_impr_change    -0.254712
pct_click_change   -0.176448
pos_change          0.036884
activity_change     0.057715
log_imp_prev        0.452082
Name: high_confidence_decline, dtype: float64

Any feature above 0.95 correlation (near-duplicate of label): none

ID columns present in FEATURES: none — clean


**Observed result:** [fill in from output]. No feature is target-derived, no
forward-looking columns exist in the frame, `fact_query_90d` remains excluded for the
same reasoning as Week 4/5, and no ID column leaked into the feature set. [If the
correlation check surfaces anything above ~0.95, name it here and explain — otherwise
state plainly that nothing crossed that threshold.] This audit doesn't prove there is no
leakage of any kind (e.g. a feature could still be leaky in a way this checklist doesn't
catch), only that the specific leakage patterns checked for are absent — a directional
finding, not a certification.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



My boldest sentence from Week 5 (Section 3 markdown): *"the model separates the two
classes better across the full ranked list"* stated as an unqualified fact, and the
Week-5 Section 1 rationale that the model "directly tests whether volume... separates
real declines from floor noise" — phrased as if the test settles the question outright.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
original = (
    "The model separates the two classes better across the full ranked list."
)
rewritten = (
    "On this held-out, client-grouped test set, the model's PR-AUC was measured "
    "at a higher value than the rule's — an observed, directional result on this "
    "particular target and split, not a claim that the model generalizes to unseen "
    "clients or future time periods without further validation."
)
print("ORIGINAL:", original)
print("\nREWRITTEN:", rewritten)

ORIGINAL: The model separates the two classes better across the full ranked list.

REWRITTEN: On this held-out, client-grouped test set, the model's PR-AUC was measured at a higher value than the rule's — an observed, directional result on this particular target and split, not a claim that the model generalizes to unseen clients or future time periods without further validation.


**Rewritten claim:** "On this held-out, client-grouped test set, the model's PR-AUC was
*measured* at a higher value than the rule's — an *observed*, *directional* result on
this particular target and split, not a claim that the model *generalizes* to unseen
clients or future time periods without further validation. This stays *decision-support*
for prioritizing review, built on a proxy target (`high_confidence_decline`) derived from
Week 4's own judgment call, not a validated ground-truth outcome."

The original version implied a general, settled fact about the model's ability
("separates classes better," full stop). The rewrite scopes the claim to what was
actually measured (this test set, this split, this proxy target) and names the
limitation (proxy label, no confirmed generalization) directly in the sentence, rather
than leaving it for a footnote.

## Self-check

Before you submit, confirm each line honestly:

-  Every section above is filled — markdown thinking AND the code that backs it
-  The notebook runs top to bottom with no errors (Runtime → Run all)
-  No client names, URLs, or private queries anywhere
-  My claims use careful words: observed, measured, directional, decision-support
-  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.